In [1]:
# === composite severity with carry-forward (LOCF) ===
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("D:/PPMI_Project")
df = pd.read_csv(ROOT/"data_processed"/"clinical_longitudinal.csv", parse_dates=["INFODT"], infer_datetime_format=True)

# Keep key columns
cols_needed = ["PATNO","EVENT_ID","INFODT","AGE","SEX","NP1TOT","NP2TOT","NP3TOT","MCATOT","Y_PD"]
df = df[cols_needed].copy()

# Sort and apply carry-forward per patient for clinical scores
df = df.sort_values(["PATNO","INFODT"])
for c in ["NP1TOT","NP2TOT","NP3TOT","MCATOT"]:
    df[c] = df.groupby("PATNO")[c].ffill()

# Drop rows that still have all clinical components missing after LOCF
mask_all_missing = df[["NP1TOT","NP2TOT","NP3TOT","MCATOT"]].isna().all(axis=1)
df = df[~mask_all_missing].copy()

# Simple cohort min-max normalization (robust to outliers via clipping at 1st–99th pct)
def robust_minmax(s):
    lo = s.quantile(0.01)
    hi = s.quantile(0.99)
    s_clip = s.clip(lo, hi)
    return (s_clip - lo) / (hi - lo + 1e-9)

n_np1 = robust_minmax(df["NP1TOT"])
n_np2 = robust_minmax(df["NP2TOT"])
n_np3 = robust_minmax(df["NP3TOT"])
n_mca = robust_minmax(df["MCATOT"])

# Composite severity (equal weights). MoCA is inverse (higher=better), so use (1 - normalized MoCA)
df["SEVERITY"] = (n_np1 + n_np2 + n_np3 + (1.0 - n_mca)) / 4.0

# Clean AGE/SEX
df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
df["SEX"] = pd.to_numeric(df["SEX"], errors="coerce").clip(0,1)

# Save enriched file
outpath = ROOT/"data_processed"/"clinical_with_severity.csv"
df.to_csv(outpath, index=False)
print("Saved ->", outpath)
print("Rows:", len(df))
print(df[["SEVERITY"]].describe())

Saved -> D:\PPMI_Project\data_processed\clinical_with_severity.csv
Rows: 13010
           SEVERITY
count  9.606000e+03
mean   2.074378e-01
std    1.547696e-01
min    1.785713e-11
25%    9.205699e-02
50%    1.743218e-01
75%    2.849727e-01
max    9.918033e-01


C:\Users\sahil\AppData\Local\Temp\ipykernel_3484\1580650823.py:7: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df = pd.read_csv(ROOT/"data_processed"/"clinical_longitudinal.csv", parse_dates=["INFODT"], infer_datetime_format=True)
